# Orpheus Engine: Audio Analysis with MLflow

This notebook demonstrates how to use the Orpheus Engine's audio analysis capabilities with MLflow integration. You'll learn how to:

1. Load and preprocess audio files
2. Visualize waveforms and spectrograms
3. Extract audio features
4. Track experiments using MLflow
5. Apply machine learning models for audio classification

Let's start by setting up the environment and importing the necessary libraries.

In [ ]:
# Import required libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import soundfile as sf
import mlflow
import mlflow.sklearn
import warnings

from pathlib import Path
from IPython.display import Audio, display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Configure matplotlib for better visualization
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
warnings.filterwarnings('ignore')

# Set up MLflow tracking
os.environ['MLFLOW_TRACKING_URI'] = 'http://localhost:5002'
os.environ['MLFLOW_EXPERIMENT_NAME'] = 'orpheus-audio-analysis'

# Create a data directory if it doesn't exist
data_dir = Path('../data/samples')
if not data_dir.exists():
    data_dir.mkdir(parents=True, exist_ok=True)
    print(f"Created data directory at {data_dir}")
else:
    print(f"Data directory exists at {data_dir}")

print("Environment set up successfully!")
print(f"Python version: {sys.version}")
print(f"Librosa version: {librosa.__version__}")
print(f"MLflow version: {mlflow.__version__}")

## Loading and Visualizing Audio Data

Let's start by creating a few utility functions to load audio files and visualize their waveforms and spectrograms. If you don't have any audio files available, we'll generate a synthetic audio sample for demonstration purposes.

In [ ]:
# Define utility functions for loading and visualizing audio
def load_audio(file_path, sr=22050):
    """Load an audio file and return the signal and sample rate"""
    try:
        y, sr = librosa.load(file_path, sr=sr)
        return y, sr
    except Exception as e:
        print(f"Error loading audio file: {e}")
        return None, None

def generate_synthetic_audio(duration=5, sr=22050):
    """Generate a synthetic audio sample with multiple frequency components"""
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    
    # Generate a chord with multiple frequencies (C major: C, E, G)
    c_freq = 261.63  # C4 frequency
    e_freq = 329.63  # E4 frequency
    g_freq = 392.00  # G4 frequency
    
    # Create a signal with these frequencies and some amplitude modulation
    y = 0.3 * np.sin(2 * np.pi * c_freq * t)
    y += 0.2 * np.sin(2 * np.pi * e_freq * t)
    y += 0.2 * np.sin(2 * np.pi * g_freq * t)
    
    # Add some amplitude modulation
    y *= 0.5 + 0.5 * np.sin(2 * np.pi * 0.25 * t)
    
    # Add a subtle attack and decay
    attack = np.linspace(0, 1, int(0.1 * sr))
    decay = np.linspace(1, 0, int(0.5 * sr))
    y[:len(attack)] *= attack
    y[-len(decay):] *= decay
    
    return y, sr

def plot_waveform(y, sr, title="Waveform"):
    """Plot the waveform of an audio signal"""
    plt.figure(figsize=(14, 5))
    librosa.display.waveshow(y, sr=sr)
    plt.title(title)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.tight_layout()
    plt.show()

def plot_spectrogram(y, sr, title="Spectrogram"):
    """Plot the spectrogram of an audio signal"""
    D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    plt.figure(figsize=(14, 5))
    librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='log')
    plt.colorbar(format='%+2.0f dB')
    plt.title(title)
    plt.xlabel("Time (s)")
    plt.ylabel("Frequency (Hz)")
    plt.tight_layout()
    plt.show()

def plot_mel_spectrogram(y, sr, title="Mel Spectrogram"):
    """Plot the mel spectrogram of an audio signal"""
    S = librosa.feature.melspectrogram(y=y, sr=sr)
    S_dB = librosa.power_to_db(S, ref=np.max)
    plt.figure(figsize=(14, 5))
    librosa.display.specshow(S_dB, sr=sr, x_axis='time', y_axis='mel')
    plt.colorbar(format='%+2.0f dB')
    plt.title(title)
    plt.xlabel("Time (s)")
    plt.ylabel("Mel Frequency")
    plt.tight_layout()
    plt.show()

def plot_chromagram(y, sr, title="Chromagram"):
    """Plot the chromagram of an audio signal"""
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    plt.figure(figsize=(14, 5))
    librosa.display.specshow(chroma, sr=sr, x_axis='time', y_axis='chroma')
    plt.colorbar()
    plt.title(title)
    plt.xlabel("Time (s)")
    plt.ylabel("Pitch Class")
    plt.tight_layout()
    plt.show()

def visualize_audio(y, sr, title_prefix="Audio"):
    """Visualize an audio signal with multiple plots"""
    plot_waveform(y, sr, title=f"{title_prefix} - Waveform")
    plot_spectrogram(y, sr, title=f"{title_prefix} - Spectrogram")
    plot_mel_spectrogram(y, sr, title=f"{title_prefix} - Mel Spectrogram")
    plot_chromagram(y, sr, title=f"{title_prefix} - Chromagram")
    
    # Display audio player
    display(Audio(y, rate=sr))

# Check for existing audio files or generate synthetic audio
audio_files = list(data_dir.glob('*.wav'))
if audio_files:
    print(f"Found {len(audio_files)} audio files in {data_dir}")
    audio_file = audio_files[0]
    print(f"Using file: {audio_file}")
    y, sr = load_audio(audio_file)
    if y is not None:
        title = f"Audio File: {audio_file.name}"
    else:
        print("Generating synthetic audio instead")
        y, sr = generate_synthetic_audio()
        title = "Synthetic Audio"
else:
    print(f"No audio files found in {data_dir}. Generating synthetic audio.")
    y, sr = generate_synthetic_audio()
    title = "Synthetic Audio"
    
    # Save the synthetic audio
    synthetic_path = data_dir / "synthetic_audio.wav"
    sf.write(synthetic_path, y, sr)
    print(f"Saved synthetic audio to {synthetic_path}")

# Visualize the audio
visualize_audio(y, sr, title_prefix=title)

## Audio Feature Extraction

Now let's extract audio features that are commonly used in audio analysis and machine learning tasks. We'll extract both temporal features (like zero-crossing rate) and spectral features (like MFCCs, spectral centroid, etc.).

In [ ]:
# Define function to extract audio features
def extract_features(y, sr):
    """Extract common audio features from an audio signal"""
    features = {}
    
    # Basic statistics
    features['duration'] = len(y) / sr
    features['rms'] = np.sqrt(np.mean(y**2))
    
    # Temporal features
    zero_crossings = librosa.zero_crossings(y)
    features['zero_crossing_rate'] = sum(zero_crossings) / len(zero_crossings)
    
    # Spectral features - aggregated by mean and std
    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    features['spectral_centroid_mean'] = np.mean(spectral_centroid)
    features['spectral_centroid_std'] = np.std(spectral_centroid)
    
    spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    features['spectral_bandwidth_mean'] = np.mean(spectral_bandwidth)
    features['spectral_bandwidth_std'] = np.std(spectral_bandwidth)
    
    spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
    features['spectral_rolloff_mean'] = np.mean(spectral_rolloff)
    features['spectral_rolloff_std'] = np.std(spectral_rolloff)
    
    # MFCCs
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    for i in range(13):
        features[f'mfcc{i+1}_mean'] = np.mean(mfccs[i])
        features[f'mfcc{i+1}_std'] = np.std(mfccs[i])
    
    # Chroma features
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    features['chroma_mean'] = np.mean(chroma)
    features['chroma_std'] = np.std(chroma)
    
    # Tempo and beat features
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    features['tempo'] = tempo
    
    # Harmony features
    harmony, _ = librosa.effects.hpss(y)
    features['harmony_mean'] = np.mean(harmony)
    features['harmony_std'] = np.std(harmony)
    
    return features

# Extract features from the audio sample
features = extract_features(y, sr)

# Display features in a DataFrame
features_df = pd.DataFrame(features.items(), columns=['Feature', 'Value'])
features_df = features_df.sort_values('Feature')

# Display features
print("Extracted Audio Features:")
features_df

## Feature Visualization

Let's visualize some of the extracted features to better understand the audio characteristics. We'll create plots for temporal features over time and distributions of spectral features.

In [ ]:
# Visualize temporal features
def plot_feature_over_time(y, sr, feature_name, feature_func, hop_length=512):
    """Plot a feature over time"""
    # Calculate the feature
    feature = feature_func(y=y, sr=sr, hop_length=hop_length)[0]
    
    # Create time axis
    times = librosa.times_like(feature, sr=sr, hop_length=hop_length)
    
    # Plot
    plt.figure(figsize=(14, 5))
    plt.plot(times, feature)
    plt.title(f"{feature_name} over Time")
    plt.xlabel("Time (s)")
    plt.ylabel(feature_name)
    plt.tight_layout()
    plt.show()
    
    return feature

# Plot spectral centroid over time
spectral_centroid = plot_feature_over_time(
    y, sr, "Spectral Centroid", librosa.feature.spectral_centroid
)

# Plot spectral rolloff over time
spectral_rolloff = plot_feature_over_time(
    y, sr, "Spectral Rolloff", librosa.feature.spectral_rolloff
)

# Plot zero crossing rate over time
# We need to handle this differently because of the function signature
hop_length = 512
zero_crossings = librosa.feature.zero_crossing_rate(y, hop_length=hop_length)[0]
times = librosa.times_like(zero_crossings, sr=sr, hop_length=hop_length)
plt.figure(figsize=(14, 5))
plt.plot(times, zero_crossings)
plt.title("Zero Crossing Rate over Time")
plt.xlabel("Time (s)")
plt.ylabel("Zero Crossing Rate")
plt.tight_layout()
plt.show()

# Plot MFCCs
mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
plt.figure(figsize=(14, 5))
librosa.display.specshow(mfccs, sr=sr, x_axis='time')
plt.colorbar(format='%+2.0f')
plt.title('MFCCs')
plt.xlabel("Time (s)")
plt.ylabel("MFCC Coefficients")
plt.tight_layout()
plt.show()

# Visualize feature distributions
def plot_feature_distributions(features_df):
    """Plot distributions of numerical features"""
    # Select numeric features
    numeric_features = features_df.copy()
    
    # Create a figure with subplots
    n_features = len(numeric_features)
    n_cols = 2
    n_rows = (n_features + n_cols - 1) // n_cols
    
    plt.figure(figsize=(14, n_rows * 3))
    
    for i, (feature, value) in enumerate(numeric_features.values):
        plt.subplot(n_rows, n_cols, i + 1)
        plt.bar([feature], [value], color='skyblue')
        plt.title(f"{feature}")
        plt.xticks(rotation=45)
        plt.subplots_adjust(hspace=0.5)
    
    plt.tight_layout()
    plt.show()

# Filter out features for visualization (selecting a subset)
plot_features = features_df[features_df['Feature'].str.contains('mean|tempo|zero')].copy()
plot_feature_distributions(plot_features)

## MLflow Integration for Experiment Tracking

Now let's demonstrate how to use MLflow to track experiments with audio analysis. MLflow provides a way to log parameters, metrics, and artifacts from your audio processing and machine learning workflows.

First, we'll set up an MLflow experiment and demonstrate logging features, visualizations, and audio samples.

In [ ]:
# Set up MLflow experiment
experiment_name = "orpheus-audio-analysis"
mlflow.set_experiment(experiment_name)

# Define function to log audio features to MLflow
def log_audio_analysis(audio_path, y, sr, features, run_name="Audio Analysis"):
    """Log audio analysis results to MLflow"""
    with mlflow.start_run(run_name=run_name):
        # Log parameters
        mlflow.log_param("sample_rate", sr)
        mlflow.log_param("duration", features['duration'])
        mlflow.log_param("audio_file", audio_path if isinstance(audio_path, str) else "synthetic_audio")
        
        # Log metrics (features)
        for feature, value in features.items():
            # Skip non-numeric features
            if isinstance(value, (int, float)):
                mlflow.log_metric(feature, value)
        
        # Create and log visualizations as artifacts
        artifact_dir = "artifacts"
        os.makedirs(artifact_dir, exist_ok=True)
        
        # Log waveform
        plt.figure(figsize=(10, 4))
        librosa.display.waveshow(y, sr=sr)
        plt.title("Waveform")
        plt.tight_layout()
        waveform_path = os.path.join(artifact_dir, "waveform.png")
        plt.savefig(waveform_path)
        plt.close()
        mlflow.log_artifact(waveform_path)
        
        # Log spectrogram
        plt.figure(figsize=(10, 4))
        D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
        librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='log')
        plt.colorbar(format='%+2.0f dB')
        plt.title("Spectrogram")
        plt.tight_layout()
        spectrogram_path = os.path.join(artifact_dir, "spectrogram.png")
        plt.savefig(spectrogram_path)
        plt.close()
        mlflow.log_artifact(spectrogram_path)
        
        # Log MFCCs
        plt.figure(figsize=(10, 4))
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        librosa.display.specshow(mfccs, sr=sr, x_axis='time')
        plt.colorbar(format='%+2.0f')
        plt.title('MFCCs')
        plt.tight_layout()
        mfccs_path = os.path.join(artifact_dir, "mfccs.png")
        plt.savefig(mfccs_path)
        plt.close()
        mlflow.log_artifact(mfccs_path)
        
        # Log the audio file
        audio_out_path = os.path.join(artifact_dir, "audio_sample.wav")
        sf.write(audio_out_path, y, sr)
        mlflow.log_artifact(audio_out_path)
        
        # Log features as a CSV
        features_df_path = os.path.join(artifact_dir, "features.csv")
        features_df.to_csv(features_df_path, index=False)
        mlflow.log_artifact(features_df_path)
        
        # Get the run ID for display
        run_id = mlflow.active_run().info.run_id
        
    return run_id

# Log the current audio analysis to MLflow
audio_path = str(data_dir / "synthetic_audio.wav") if not audio_files else str(audio_files[0])
run_id = log_audio_analysis(audio_path, y, sr, features)

print(f"✅ Audio analysis logged to MLflow experiment '{experiment_name}'")
print(f"Run ID: {run_id}")
print("\nYou can view this run in the MLflow UI with:")
print(f"mlflow ui --port 5002")
print("Then open http://localhost:5002 in your browser")

## Audio Classification with Machine Learning

Now let's demonstrate how to use machine learning for audio classification using the extracted features. We'll simulate a dataset of audio features for this demonstration, since we may not have multiple audio files available.

In [ ]:
# Generate synthetic dataset for demonstration
def generate_synthetic_dataset(n_samples=100):
    """Generate a synthetic dataset of audio features"""
    np.random.seed(42)  # For reproducibility
    
    # Define classes
    classes = ['speech', 'music', 'noise']
    
    # Create feature names
    feature_names = [
        'duration', 'rms', 'zero_crossing_rate', 
        'spectral_centroid_mean', 'spectral_bandwidth_mean', 
        'spectral_rolloff_mean', 'tempo',
    ] + [f'mfcc{i+1}_mean' for i in range(13)]
    
    # Generate data
    data = []
    labels = []
    
    for _ in range(n_samples):
        # Randomly choose a class
        class_idx = np.random.randint(0, len(classes))
        class_label = classes[class_idx]
        labels.append(class_label)
        
        # Generate features based on the class
        features = []
        if class_label == 'speech':
            # Speech-like features
            features.append(np.random.uniform(1, 5))  # duration
            features.append(np.random.uniform(0.05, 0.2))  # rms
            features.append(np.random.uniform(0.05, 0.15))  # zcr
            features.append(np.random.uniform(1500, 2500))  # spec_centroid
            features.append(np.random.uniform(1500, 2000))  # spec_bandwidth
            features.append(np.random.uniform(3000, 5000))  # spec_rolloff
            features.append(np.random.uniform(80, 160))  # tempo
            # MFCCs
            for _ in range(13):
                features.append(np.random.normal(0, 1))
        elif class_label == 'music':
            # Music-like features
            features.append(np.random.uniform(3, 10))  # duration
            features.append(np.random.uniform(0.1, 0.3))  # rms
            features.append(np.random.uniform(0.01, 0.1))  # zcr
            features.append(np.random.uniform(2000, 3500))  # spec_centroid
            features.append(np.random.uniform(1800, 2500))  # spec_bandwidth
            features.append(np.random.uniform(4000, 7000))  # spec_rolloff
            features.append(np.random.uniform(60, 180))  # tempo
            # MFCCs
            for _ in range(13):
                features.append(np.random.normal(2, 1))
        else:  # noise
            # Noise-like features
            features.append(np.random.uniform(0.5, 3))  # duration
            features.append(np.random.uniform(0.2, 0.4))  # rms
            features.append(np.random.uniform(0.2, 0.5))  # zcr
            features.append(np.random.uniform(3000, 5000))  # spec_centroid
            features.append(np.random.uniform(3000, 5000))  # spec_bandwidth
            features.append(np.random.uniform(6000, 10000))  # spec_rolloff
            features.append(np.random.uniform(20, 200))  # tempo
            # MFCCs
            for _ in range(13):
                features.append(np.random.normal(-2, 2))
        
        # Add some noise
        features = [f + np.random.normal(0, 0.1) for f in features]
        data.append(features)
    
    # Create DataFrame
    df = pd.DataFrame(data, columns=feature_names)
    df['class'] = labels
    
    return df, feature_names

# Generate synthetic dataset
df, feature_names = generate_synthetic_dataset(n_samples=100)

print("Generated synthetic dataset with features:")
display(df.head())

# Split into features and target
X = df[feature_names]
y = df['class']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train a Random Forest classifier with MLflow tracking
with mlflow.start_run(run_name="Audio Classification"):
    # Log parameters
    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("n_samples", len(df))
    mlflow.log_param("n_features", len(feature_names))
    
    # Train the model
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train_scaled, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test_scaled)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred, output_dict=True)
    
    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    for class_name in class_report:
        if class_name not in ['accuracy', 'macro avg', 'weighted avg']:
            mlflow.log_metric(f"f1_{class_name}", class_report[class_name]['f1-score'])
            mlflow.log_metric(f"precision_{class_name}", class_report[class_name]['precision'])
            mlflow.log_metric(f"recall_{class_name}", class_report[class_name]['recall'])
    
    # Create and log visualizations
    artifact_dir = "artifacts"
    os.makedirs(artifact_dir, exist_ok=True)
    
    # Plot confusion matrix
    plt.figure(figsize=(10, 8))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=model.classes_, yticklabels=model.classes_)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    conf_matrix_path = os.path.join(artifact_dir, "confusion_matrix.png")
    plt.savefig(conf_matrix_path)
    plt.close()
    mlflow.log_artifact(conf_matrix_path)
    
    # Plot feature importance
    plt.figure(figsize=(12, 8))
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]
    plt.title('Feature Importances')
    plt.bar(range(X.shape[1]), importances[indices], align='center')
    plt.xticks(range(X.shape[1]), [feature_names[i] for i in indices], rotation=90)
    plt.tight_layout()
    importance_path = os.path.join(artifact_dir, "feature_importance.png")
    plt.savefig(importance_path)
    plt.close()
    mlflow.log_artifact(importance_path)
    
    # Log the model
    mlflow.sklearn.log_model(model, "model")
    
    # Display results
    print(f"Model training complete. Accuracy: {accuracy:.4f}")
    print("\nConfusion Matrix:")
    print(conf_matrix)
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    
    # Get run ID
    run_id = mlflow.active_run().info.run_id

print(f"\n✅ Model logged to MLflow with run_id: {run_id}")

# Plot the confusion matrix (again, for notebook display)
plt.figure(figsize=(10, 8))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=model.classes_, yticklabels=model.classes_)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

# Plot feature importance
plt.figure(figsize=(12, 8))
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]
plt.title('Feature Importances')
plt.bar(range(X.shape[1]), importances[indices], align='center')
plt.xticks(range(X.shape[1]), [feature_names[i] for i in indices], rotation=90)
plt.tight_layout()
plt.show()

## Conclusion and Next Steps

In this notebook, we've demonstrated the core audio analysis capabilities of the Orpheus Engine:

1. **Audio Loading and Visualization**: We loaded audio data and visualized waveforms, spectrograms, and other representations.

2. **Feature Extraction**: We extracted temporal and spectral features from audio signals that can be used for various applications.

3. **MLflow Integration**: We tracked our experiments, logged parameters, metrics, and artifacts using MLflow.

4. **Machine Learning for Audio**: We built a simple audio classification model using the extracted features.

### Potential Next Steps

1. **Real-time Audio Analysis**: Integrate these capabilities with the Orpheus Engine DAW for real-time analysis during recording or playback.

2. **Advanced Models**: Implement more sophisticated models like convolutional neural networks (CNNs) for audio classification or segmentation.

3. **Custom Effects**: Use the extracted features to develop intelligent audio effects that adapt to the content.

4. **User Interface Integration**: Visualize these analyses directly in the Orpheus Engine interface for an enhanced user experience.

### Resources

- [Librosa Documentation](https://librosa.org/doc/latest/index.html)
- [MLflow Documentation](https://mlflow.org/docs/latest/index.html)
- [Orpheus Engine Documentation](../docs/README.md)